In [ ]:
%%capture
%pip install lightning

In [ ]:
import h5py

from PIL import Image
from tqdm.notebook import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

from torchvision import transforms
import torchvision.models as models

import torchmetrics

import lightning as L
from lightning import LightningModule, LightningDataModule

In [5]:
BACKBONE_EPOCHS = 10
CLASSIFIER_EPOCH = 10

BATCH_SIZE = 512 
SEED = 42

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

run_name = 'reduced_dataset_10ep'

# Model

In [6]:
class ResNet18Classifier(LightningModule):
    def __init__(self, num_classes: int = 2):
        super().__init__()
        self.model = models.resnet18(weights='DEFAULT')
        self.model.fc = nn.Linear(self.model.fc.in_features, num_classes)
        self.num_classes = num_classes
        
        self.criterion = nn.CrossEntropyLoss()

        self.train_acc = torchmetrics.Accuracy("binary", num_classes=num_classes)
        self.val_acc = torchmetrics.Accuracy("binary", num_classes=num_classes)
        self.test_acc = torchmetrics.Accuracy("binary", num_classes=num_classes)

    def forward(self, x):
        return self.model(x)

    def _step(self, batch, stage: str):
        images, labels = batch

        outputs = self(images)
        loss = self.criterion(outputs, labels)
        preds = torch.argmax(outputs, dim=1)

        if stage == "train":
            self.train_acc(preds, labels)
        elif stage == "val":
            self.val_acc(preds, labels)
        elif stage == "test":
            self.test_acc(preds, labels)
        else:
            raise ValueError(f"Unknown stage: {stage}")
        
        self.log(f"{stage}_loss", loss, on_step=False, on_epoch=True, prog_bar=True)
        self.log(f"{stage}_acc", getattr(self, f"{stage}_acc"), on_step=False, on_epoch=True, prog_bar=True)
        
        return loss

    def training_step(self, batch, batch_idx):
        return self._step(batch, "train")
    def validation_step(self, batch, batch_idx):
        return self._step(batch, "val")
    def test_step(self, batch, batch_idx):
        return self._step(batch, "test")

    def configure_optimizers(self):
        optimizer = optim.AdamW(self.parameters(), lr=1e-3, weight_decay=0.05)
        
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='min', factor=0.1, patience=5
        )
        return {
            "optimizer": optimizer,
            "lr_scheduler": {
                "scheduler": scheduler,
                "monitor": "val_loss",
                "interval": "epoch",
                "frequency": 1,
            },
        }
        
    def freeze_backbone(self):
        for param in self.model.parameters():
            param.requires_grad = False
        for param in self.model.fc.parameters():  # Keep classifier trainable
            param.requires_grad = True

    def unfreeze_backbone(self):
        for param in self.model.parameters():
            param.requires_grad = True

# Dataset

In [7]:
class PCAMDataset(torch.utils.data.Dataset):
    def __init__(self, input_file_path: str, label_file_path: str, transform=None, lazy: bool = False, max_samples=None):
        self.input_file_path = input_file_path
        self.label_file_path = label_file_path
        self.transform = transform
        self.lazy = lazy
        self.max_samples = max_samples
        
        input_files = h5py.File(self.input_file_path)["x"]
        self.input_files = input_files
        target_files = h5py.File(self.label_file_path)["y"]
        self.target_files = target_files
        
        # Limit the dataset size
        if max_samples is not None:
            self.length = min(max_samples, len(input_files))
        else:
            self.length = len(input_files)
        
        if not lazy:
            self.images = [Image.fromarray(input_files[i]).convert("RGB") for i in tqdm(range(self.length))]
            self.targets = [int(target_files[i, 0, 0, 0]) for i in tqdm(range(self.length))]
        
    def __len__(self) -> int:
        return self.length
            
    def __getitem__(self, idx: int):
        if idx >= self.length:
            raise IndexError("Index out of range")
            
        if not self.lazy:
            image, target = self.images[idx], torch.tensor(self.targets[idx])
        else:
            image = Image.fromarray(self.input_files[idx]).convert("RGB")
            target = int(self.target_files[idx, 0, 0, 0])
    
        if self.transform:
            image = self.transform(image)
        return image, target

Applied transformations

1. Random Horizontal and Vertical Flips

   - Histopathological images lack a fixed orientation; flipping helps the model learn rotational invariance, improving its ability to generalize across different tissue orientations.


2. Random Rotations (e.g., ±90°)

   - Tissues can appear at various angles; rotating images ensures the model doesn't become biased toward a specific orientation, enhancing robustness.

3. Color Jittering (Brightness, Contrast, Saturation, Hue)

    Purpose: Mimic staining variability.

    Justification: Histopathology slides often exhibit staining differences; adjusting color properties helps the model become invariant to such variations, focusing on morphological features instead.
    collab.dvb.bayern+3restack.io+3GitHub+3

4. Random Erasing

    Purpose: Simulate occlusions.

    Justification: Randomly erasing parts of an image forces the model to rely on the surrounding context, improving its ability to handle occlusions and incomplete data.

In [8]:
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(degrees=90),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.RandomErasing(p=0.5, scale=(0.02, 0.2)),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])



# Transform for validation and test - just rescaling and normalization
eval_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

In [9]:
train_data = PCAMDataset(
    input_file_path="/kaggle/input/metastatic-tissue-classification-patchcamelyon/pcam/training_split.h5",
    label_file_path="/kaggle/input/metastatic-tissue-classification-patchcamelyon/Labels/Labels/camelyonpatch_level_2_split_train_y.h5",
    transform=train_transform,
    lazy=False,
    max_samples=None
)

  0%|          | 0/262144 [00:00<?, ?it/s]

  0%|          | 0/262144 [00:00<?, ?it/s]

In [10]:
val_data = PCAMDataset(
    input_file_path="/kaggle/input/metastatic-tissue-classification-patchcamelyon/pcam/validation_split.h5",
    label_file_path="/kaggle/input/metastatic-tissue-classification-patchcamelyon/Labels/Labels/camelyonpatch_level_2_split_valid_y.h5",
    transform=eval_transform,
    lazy=False,
    max_samples=None
)

  0%|          | 0/32768 [00:00<?, ?it/s]

  0%|          | 0/32768 [00:00<?, ?it/s]

In [11]:
test_data = PCAMDataset(
    input_file_path="/kaggle/input/metastatic-tissue-classification-patchcamelyon/pcam/test_split.h5",
    label_file_path="/kaggle/input/metastatic-tissue-classification-patchcamelyon/Labels/Labels/camelyonpatch_level_2_split_test_y.h5",
    transform=eval_transform,
    lazy=False,
    max_samples=None
)

  0%|          | 0/32768 [00:00<?, ?it/s]

  0%|          | 0/32768 [00:00<?, ?it/s]

# Data Module

In [12]:
class PCAMDataModule(LightningDataModule):
    def __init__(self, train, val, test=None, name=""):
        super().__init__()
        self.train = train
        self.val = val
        self.test = test
        
        self.name = name

    def setup(self, stage: str = None):
        if stage == "fit" or stage is None:
            self.train_dataset = self.train
            self.val_dataset = self.val
        if stage == "test" or stage is None:
            if self.test is None:
                raise ValueError("Test dataset is not provided.")
            else:
                self.test_dataset = self.test

    def train_dataloader(self):
        return DataLoader(self.train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)

    def val_dataloader(self):
        return DataLoader(self.val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

    def test_dataloader(self):
        return DataLoader(self.test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

In [13]:
model = ResNet18Classifier(num_classes=2)

In [14]:
model

ResNet18Classifier(
  (model): ResNet(
    (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, 

In [15]:
datamodule = PCAMDataModule(
    train=train_data,
    val=val_data,
    test=test_data
)

In [16]:
ckbs = [
    L.pytorch.callbacks.ModelCheckpoint(
            dirpath="/kaggle/working/checkpoints/",
            monitor="val_loss",
            mode="min",
            save_top_k=1,
            filename="{epoch:02d}-{val_loss:.2f}",
        ),
    L.pytorch.callbacks.LearningRateMonitor(logging_interval='epoch'),
    L.pytorch.callbacks.RichProgressBar(),
    L.pytorch.callbacks.early_stopping.EarlyStopping(
        monitor="val_loss", min_delta=0.00, patience=5, verbose=False, mode="min"
    )
]

In [17]:
L.seed_everything(SEED, workers=True)

INFO: Seed set to 42


42

In [17]:
model.freeze_backbone()

trainer = L.Trainer(
    max_epochs=BACKBONE_EPOCHS,
    callbacks=ckbs,
    precision="16-mixed",
    log_every_n_steps=10,
)

trainer.fit(model, datamodule=datamodule)

INFO: Using 16bit Automatic Mixed Precision (AMP)
INFO: GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO: HPU available: False, using: 0 HPUs
2025-06-08 13:03:49.844610: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1749387829.866236     389 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1749387829.872861     389 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┓
┃   ┃ Name      ┃ Type             ┃ Params ┃ Mode  ┃
┡━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━┩
│ 0 │ model     │ ResNet           │ 11.2 M │ train │
│ 1 │ criterion │ CrossEntropyLoss │      0 │ train │
│ 2 │ train_acc │ BinaryAccuracy   │      0 │ train │
│ 3 │ val_acc   │ BinaryAccuracy   │      0 │ train │
│ 4 │ test_acc  │ BinaryAccuracy   │      0 │ train │
└───┴───────────┴──────────────────┴────────┴───────┘

Trainable params: 1.0 K                                                                                            
Non-trainable params: 11.2 M                                                                                       
Total params: 11.2 M                                                                                               
Total estimated model params size (MB): 44                                                                         
Modules in train mode: 72                                                                                          
Modules in eval mode: 0

Output()

INFO: `Trainer.fit` stopped: `max_epochs=10` reached.


In [18]:
trainer.test(model, datamodule=datamodule, ckpt_path="best")

INFO: Restoring states from the checkpoint path at /kaggle/working/checkpoints/epoch=05-val_loss=0.48.ckpt
INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO: Loaded model weights from the checkpoint at /kaggle/working/checkpoints/epoch=05-val_loss=0.48.ckpt


Output()

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_acc          │     0.77667236328125      │
│         test_loss         │    0.4821639358997345     │
└───────────────────────────┴───────────────────────────┘

[{'test_loss': 0.4821639358997345, 'test_acc': 0.77667236328125}]

In [19]:
# After training is complete
best_model_path = trainer.checkpoint_callback.best_model_path
print(f"Best model saved at: {best_model_path}")

Best model saved at: /kaggle/working/checkpoints/epoch=05-val_loss=0.48.ckpt


In [20]:
# Load the best model
best_model = ResNet18Classifier.load_from_checkpoint(best_model_path)
best_model.eval()  # Set to evaluation mode

ResNet18Classifier(
  (model): ResNet(
    (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, 

In [ ]:
# Or if you want to continue training with the best weights
model = ResNet18Classifier.load_from_checkpoint(best_model_path)

In [ ]:
model.unfreeze_backbone()

trainer = L.Trainer(
    max_epochs=CLASSIFIER_EPOCH,
    callbacks=ckbs,
    precision="16-mixed",
    log_every_n_steps=10
)

trainer.fit(model, datamodule=datamodule)

INFO: Using 16bit Automatic Mixed Precision (AMP)
INFO: GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO: HPU available: False, using: 0 HPUs


In [ ]:
trainer.test(model, datamodule=datamodule, ckpt_path="best")

2025-06-08 14:23:51.042766: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1749392631.064469    1182 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1749392631.071220    1182 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
INFO: Restoring states from the checkpoint path at /kaggle/working/checkpoints/epoch=06-val_loss=0.29.ckpt
INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO: Loaded model weights from the checkpoint at /kaggle/working/checkpoints/epoch=06-val_loss=0.29.ckpt


Output()

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_acc          │     0.846893310546875     │
│         test_loss         │    0.4176584780216217     │
└───────────────────────────┴───────────────────────────┘

[{'test_loss': 0.4176584780216217, 'test_acc': 0.846893310546875}]

# export the weight on a file

In [23]:
best_model_path = trainer.checkpoint_callback.best_model_path
print(f"Loading best model from: {best_model_path}")

# Load the best model
best_model = ResNet18Classifier.load_from_checkpoint(best_model_path)

# Export just the model weights (state_dict)
path = '/kaggle/working/' + run_name + ".pth"  

torch.save(best_model.model.state_dict(), path)
print("Model weights saved to: /kaggle/working/best_resnet18_weights.pth")

Loading best model from: /kaggle/working/checkpoints/epoch=06-val_loss=0.29.ckpt
Model weights saved to: /kaggle/working/best_resnet18_weights.pth


In [24]:
# The best checkpoint is already saved automatically by ModelCheckpoint
# You can copy it to a specific location with a cleaner name
import shutil

best_model_path = trainer.checkpoint_callback.best_model_path
#final_checkpoint_path = "/kaggle/working/best_resnet18_weights.ckpt"
path = '/kaggle/working/' + run_name + ".ckpt"  

# Copy the best checkpoint to your desired location
shutil.copy2(best_model_path, path)
print(f"Best checkpoint copied to: {path}")

Best checkpoint copied to: /kaggle/working/reduced_dataset_10ep.ckpt
